# Lecture 1 · From Bedrock Agents to Bedrock AgentCore

**Course:** *AI Fundamentals for Beginners: Learn LLM, Agentic AI & MCP*

In the earlier version of this lecture we built an AIOps agent with **Amazon Bedrock Agents** — the one where you clicked through the console, uploaded two OpenAPI JSON files, and wired up two Lambda functions. That product is now **Bedrock Agents Classic** and is in maintenance mode: AWS closed it to new customers on **30 July 2026** and points everyone at **Amazon Bedrock AgentCore** instead.

So we are going to rebuild the same agent, properly, the new way.

### What you will do in this notebook

1. Understand what actually changed — not just the product name
2. Install the Python packages
3. Confirm your AWS credentials work
4. Confirm you have access to a Claude model on Bedrock
5. Save the settings the next two notebooks will read

**Time:** about 15 minutes. **Cost:** a few cents — one small model call.

> **The three notebooks**
> `01` (this one) — concepts and setup
> `02` — build the agent and run it on your laptop
> `03` — deploy it to AgentCore Runtime and operate it

---
## 1. What actually changed

It is tempting to read this as a rename. It isn't. The **shape** of the thing changed.

**Bedrock Agents was one managed product.** You filled in a form — a model, an instructions textbox, action groups, knowledge bases, guardrails — and AWS ran the agent loop for you inside its own service. You could not see that loop. You could not step through it, unit-test it, or run it on your laptop.

**AgentCore is a set of building blocks.** You write the agent loop yourself in ordinary Python, using a framework you pick (Strands, LangGraph, Google ADK, OpenAI Agents, or your own code), and you attach only the managed pieces you need.

You trade *magic* for *control*. For a beginner that trade is a gift, because now every part of the agent is a thing you can read.

### The migration map

Everything from the old build has a home in the new one:

| The old way — Bedrock Agents | The new way — AgentCore | Why it's better for you |
| --- | --- | --- |
| Instructions typed into a console textbox | `SYSTEM_PROMPT` in `prompts.py` | It's code. It goes in git, gets reviewed, gets diffed. |
| Action group + hand-written OpenAPI JSON | A Python function with `@tool` | The type hints and docstring **are** the schema. Nothing to keep in sync. |
| One Lambda per action group | A plain function in the agent process | No Lambda hop, no cold start, no deployment package per tool. |
| `messageVersion` / `actionGroup` / `apiPath` response envelope | `return {...}` | That whole envelope is gone. |
| AWS runs the orchestration loop, invisibly | Strands runs it, in your process | You can print it, breakpoint it, and test it offline. |
| "Prepare agent", versions, aliases | `agentcore deploy` | One command, from your project directory. |
| `InvokeAgent` API | AgentCore Runtime endpoint (`POST /invocations`) | Standard HTTP. Works locally and hosted, unchanged. |
| Session state held by the managed service | Session-isolated microVM + AgentCore Memory | You choose what is remembered and for how long. |
| CloudWatch logs, and good luck | AgentCore Observability — traces and spans | You can see which tool was called, with what, and why. |

One thing did **not** change: the AWS APIs the agent calls. `cloudtrail:LookupEvents` and `ec2:StopInstances` work exactly as before. Only the wrapper around them changed.

### Before and after, in pictures

**Bedrock Agents (what we built last time)**

```
  You ──▶ InvokeAgent ──▶ ┌──────────────────────────────┐
                          │  Bedrock Agents (managed)    │
                          │  ┌────────────────────────┐  │
                          │  │ orchestration loop     │  │  ← you cannot see in here
                          │  │  + instructions        │  │
                          │  └───────────┬────────────┘  │
                          │   action group│(OpenAPI)     │
                          └───────────────┼──────────────┘
                                          ▼
                             Lambda ──▶ CloudTrail / EC2
```

**AgentCore (what we build now)**

```
  You ──▶ POST /invocations ──▶ ┌────────────────────────────────┐
                                │  AgentCore Runtime             │
                                │  (session-isolated microVM)    │
                                │  ┌──────────────────────────┐  │
                                │  │ your main.py             │  │  ← you wrote this
                                │  │  Strands Agent loop      │  │
                                │  │  + SYSTEM_PROMPT         │  │
                                │  │  + @tool functions ──────┼──┼──▶ CloudTrail / EC2
                                │  └──────────────────────────┘  │
                                └────────────────────────────────┘
                                    │
                                    └─▶ Observability (traces) · Memory · Gateway
```

The second diagram has more boxes *that you own* — and that is the whole point. The exact same `main.py` runs on your laptop and in the cloud.

### The AgentCore menu

AgentCore is not one service, it's a set. You pick what you need:

| Service | What it does | Used in this course? |
| --- | --- | --- |
| **Runtime** | Hosts your agent. Serverless, session-isolated, up to 8-hour runs. | ✅ Notebook 03 |
| **Observability** | Traces, spans, and logs for every agent turn, in CloudWatch. | ✅ Notebook 03 |
| **Memory** | Short-term (this conversation) and long-term (across sessions) memory. | 🔎 Discussed in 03 |
| **Gateway** | Turns existing APIs and Lambda functions into agent tools via MCP. | 🔎 Discussed in 03 |
| **Identity** | OAuth and API-key credentials so an agent can act as a user. | ✖ Not needed here |
| **Browser** | A managed, sandboxed browser the agent can drive. | ✖ |
| **Code Interpreter** | A sandbox for running code the agent writes. | ✖ |

You do not need to learn all of these. You need Runtime. The rest are there when a problem asks for them.

> **Note on Gateway:** if you already built the old Lambda-based agent, Gateway is your shortest migration path — it can expose those exact Lambda functions as MCP tools without rewriting them. We cover that at the end of notebook 03.

---
## 2. Before you run anything

You need:

- **An AWS account** where you are allowed to create IAM roles.
- **AWS CLI configured.** Run `aws configure sso` (recommended) or `aws configure`, then verify with `aws sts get-caller-identity`.
- **Bedrock model access.** In the [Bedrock console](https://console.aws.amazon.com/bedrock) → *Model access*, request access to an Anthropic Claude model. It is usually granted instantly.
- **CloudTrail enabled** — it is on by default for management events, so you almost certainly have it.
- **Python 3.10+** and Jupyter.
- *(Notebook 03 only)* **Node.js 20+**, for the AgentCore CLI.

**An EC2 instance is optional.** The agent has a dry-run mode, so you can do every exercise without one.

### Starting Jupyter

```bash
python3 -m venv .venv && source .venv/bin/activate
pip install jupyterlab
jupyter lab
```

On Windows PowerShell the activate line is `.venv\Scripts\Activate.ps1`.

### A word on cost

Running all three notebooks costs roughly **$0.50–$2.00**: a few hundred thousand Bedrock tokens, plus AgentCore Runtime billed per second while a request is in flight. Idle deployed agents cost nothing. Notebook 03 ends with a cleanup step — please run it.

---
## 3. Install the packages

Two packages do the work:

- **`strands-agents`** — the agent framework. It owns the loop: send the conversation to the model, notice when the model asks for a tool, run that tool, feed the result back, repeat until the model is done. This is the part Bedrock Agents used to hide from you.
- **`bedrock-agentcore`** — the AWS SDK for AgentCore. It gives you `BedrockAgentCoreApp`, which is the HTTP contract AgentCore Runtime speaks, and clients for Memory, Identity, and the rest.

`boto3` you already know — it's how the tools talk to CloudTrail and EC2.

In [ ]:
%pip install -q --upgrade "strands-agents>=1.0.0" "bedrock-agentcore>=0.1.0" "boto3>=1.35.0"
print("Installed. If this was your first install, restart the kernel now:")
print("  Kernel -> Restart Kernel, then continue from the next cell.")

---
## 4. Check your AWS credentials

Before anything else, prove that this notebook can talk to AWS as the identity you expect. `sts:GetCallerIdentity` is the cheapest possible way to ask "who am I?" — every AWS identity is allowed to call it, and it changes nothing.

**Edit `REGION` below** if you work somewhere other than `us-east-1`. Use a region where Bedrock offers Claude models.

In [ ]:
import boto3
import botocore

REGION = "us-east-1"          # <-- change this if you work in another region

session = boto3.Session(region_name=REGION)

try:
    who = session.client("sts").get_caller_identity()
    print("Account :", who["Account"])
    print("Identity:", who["Arn"])
    print("Region  :", session.region_name)
    print("\nCredentials OK.")
except botocore.exceptions.NoCredentialsError:
    print("No AWS credentials found.")
    print("Fix: run `aws configure sso` (then `aws sso login`), or `aws configure`.")
except botocore.exceptions.ClientError as err:
    code_ = err.response["Error"]["Code"]
    print("Credentials were found but AWS rejected them:", code_)
    if "ExpiredToken" in code_:
        print("Fix: your SSO session expired. Run `aws sso login` and restart the kernel.")

> **If this failed:** the notebook inherits credentials from your shell environment. If you ran `aws sso login` *after* starting Jupyter, Jupyter did not see it — restart Jupyter, not just the kernel.

---
## 5. Find a model you can actually use

Two things can go wrong here, and beginners hit both:

**1. You haven't enabled model access.** Bedrock does not give you Anthropic models by default. You request them once, per account, in the console.

**2. You used a base model ID where an inference profile ID was needed.** Modern Claude models on Bedrock are served through **cross-region inference profiles**. The ID looks like `us.anthropic.claude-sonnet-5` — note the `us.` prefix. The bare `anthropic.claude-sonnet-5` will fail with a `ValidationException` in most regions. The prefix tells Bedrock which pool of regions may serve your request:

- `us.` — routed within the US and Canada
- `eu.` — routed within the EU
- `global.` — routed anywhere, cheapest to get capacity, no data-residency guarantee

The cell below lists the Claude profiles your account can see.

In [ ]:
bedrock = session.client("bedrock")

try:
    profiles = bedrock.list_inference_profiles()["inferenceProfileSummaries"]
    claude = sorted(
        (p for p in profiles if "claude" in p["inferenceProfileId"].lower()),
        key=lambda p: p["inferenceProfileId"],
    )
    if claude:
        print(f"Claude inference profiles available in {REGION}:\n")
        for p in claude:
            print(f"  {p['inferenceProfileId']:<50} {p['status']}")
        print(f"\n{len(claude)} found. Copy one into MODEL_ID in the next cell.")
    else:
        print("No Claude profiles visible.")
        print("Fix: Bedrock console -> Model access -> request access to Anthropic models.")
except botocore.exceptions.ClientError as err:
    print("Could not list inference profiles:", err.response["Error"]["Code"])
    print("Your identity may be missing bedrock:ListInferenceProfiles.")

### Which one should you pick?

- **Claude Sonnet** — the default choice. Strong at multi-step tool use, which is exactly what an agent does.
- **Claude Haiku** — faster and cheaper. Fine for this demo; it will occasionally need a clearer prompt.
- **Claude Opus** — more capable than you need here, and more expensive.

Pick a Sonnet profile from the list above. If the list showed something newer than what's hard-coded below, use that instead.

In [ ]:
MODEL_ID = "us.anthropic.claude-sonnet-5"     # <-- paste your chosen ID here

runtime = session.client("bedrock-runtime")

try:
    response = runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "Reply with exactly: AgentCore setup OK"}]}],
        inferenceConfig={"maxTokens": 32},
    )
    print("Model replied:", response["output"]["message"]["content"][0]["text"])
    print("Token usage  :", response["usage"])
    print("\nYou have working model access. This is the last thing that had to be true.")
except botocore.exceptions.ClientError as err:
    code_ = err.response["Error"]["Code"]
    print(f"Model call failed: {code_}\n{err.response['Error']['Message']}\n")
    if code_ == "AccessDeniedException":
        print("-> Bedrock console -> Model access -> request access to this model.")
    elif code_ == "ValidationException":
        print("-> The model ID is wrong for this region. Use an ID from the list above,")
        print("   including its us./eu./global. prefix.")
    elif code_ == "ThrottlingException":
        print("-> Rate limited. Wait a few seconds and rerun this cell.")

> **What is `converse`?** It's Bedrock's model-agnostic chat API — the same call shape works for Claude, Nova, Llama, and the rest. You will not call it directly again in this course; Strands calls it for you. We used it here purely as a one-line proof that your access works.

---
## 6. Save your settings

Notebooks 02 and 03 read this file, so you only choose your region and model once.

In [ ]:
import json
from pathlib import Path

settings = {"region": REGION, "model_id": MODEL_ID}
Path("course_settings.json").write_text(json.dumps(settings, indent=2))

print("Saved to course_settings.json:")
print(json.dumps(settings, indent=2))

---
## 7. Troubleshooting

| Symptom | Cause | Fix |
| --- | --- | --- |
| `NoCredentialsError` | Jupyter can't see your AWS config | `aws sso login`, then restart **Jupyter itself**, not just the kernel |
| `ExpiredTokenException` | SSO session timed out | `aws sso login`, restart Jupyter |
| `AccessDeniedException` on the model call | Model access not requested | Bedrock console → *Model access* → request Anthropic models |
| `ValidationException: model identifier is invalid` | Base ID used instead of a profile ID | Use the `us.`-prefixed ID from the list |
| `ThrottlingException` | Too many calls too fast | Wait, then retry. New accounts have low Bedrock quotas |
| `ModuleNotFoundError: strands` | Kernel started before the install | Restart the kernel and rerun from the install cell |
| Model list is empty | Bedrock unavailable in that region | Try `us-east-1` or `us-west-2` |

---
## Recap

- Bedrock Agents Classic is closed to new customers; **AgentCore** is the path forward.
- The change is structural: a **managed product** became a **set of building blocks** with your code in the middle.
- Every old concept maps onto a new one — instructions become a prompt in a file, OpenAPI action groups become `@tool` functions, the invisible loop becomes a Strands loop you can read.
- Your environment is ready: credentials work, model access works, settings are saved.

**Next:** `02_build_and_test_locally.ipynb` — write the tools, assemble the agent, and watch it reason through a real AWS question on your own laptop.